# Pallas 与三类 profiling 区间

对应 kickoff R03/R12。真实 Jupyter 执行的 CPU 数值对照、不可变 capture 复查和 IR 检查。
所有 native 结果保留 **VERSION-SKEW**；设备部分仅观察开放源码 TPU lowering，不声称 TPU 编译或执行。
详见 [profiling.md](profiling.md) 和 [pallas-comparison.md](pallas-comparison.md)。

In [1]:
from pathlib import Path
import sys, json
root = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "upstream-sources.lock").is_file())
sys.path.insert(0, str(root / "research/software-stack"))
from verify_extensions import verify_pallas, verify_host, verify_compiler, verify_mosaic
base = root / "artifacts/jax-stack"
results = {"pallas": verify_pallas(base / "pallas-matmul-002"),
           "host": verify_host(base / "profile-events-001"),
           "compiler": verify_compiler(base / "compiler-events-001"),
           "mosaic": verify_mosaic(base / "mosaic-events-001")}
print({name: r["artifact_count"] for name, r in results.items()})

{'pallas': 26, 'host': 9, 'compiler': 13, 'mosaic': 14}


## 同一输入：普通 JAX 与 generic Pallas

这里重新执行小 matmul。`interpret=True` 是 generic CPU 解释器；typed TPU interpreter 的六个 grid 点来自上一单元已经复查的独立 capture。

In [2]:
import contextlib, io
import jax
import jax.numpy as jnp
import numpy as np
from pallas_probe import make_tiled_matmul
with np.load(base / "pallas-matmul-002/inputs.npz", allow_pickle=False) as inputs:
    a, w = inputs["a"], inputs["w"]
reference = a.astype(np.float64) @ w.astype(np.float64)
regular = np.asarray(jax.jit(lambda a, w: a @ w)(a, w))
with contextlib.redirect_stdout(io.StringIO()):
    tiled = np.asarray(jax.jit(make_tiled_matmul(4, 8, 6, mode=True))(a, w))
for value in (regular, tiled):
    np.testing.assert_allclose(value, reference, rtol=2e-5, atol=2e-5)
print({"shape": tiled.shape, "regular_error": float(np.max(abs(regular-reference))),
       "pallas_error": float(np.max(abs(tiled-reference)))})
print(results["pallas"]["grid_points"])

An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.


{'shape': (4, 6), 'regular_error': 5.102698930059546e-08, 'pallas_error': 5.102698930059546e-08}
[{'coordinates': [1, 0], 'core': 0}, {'coordinates': [1, 2], 'core': 0}, {'coordinates': [1, 1], 'core': 0}, {'coordinates': [0, 0], 'core': 0}, {'coordinates': [0, 2], 'core': 0}, {'coordinates': [0, 1], 'core': 0}]


## Host 生命周期

构造 `TraceAnnotation` 即开始；`__enter__` 没有重新计时。这里只输出已复查的捕获统计。
Inclusive duration 不可跨嵌套或并发范围直接相加当作总经过时间。

In [3]:
host = results["host"]["summary"]
print(host["assertions"])
print(host["event_counts"])
print(host["step_scopes"])

{'constructor_starts_before_enter': True, 'manual_span_contains_work': True, 'exception_scope_closed': True, 'outside_session_event_absent': True, 'three_step_scopes_contain_dispatch_and_wait': True}
{'research_step': 3, 'research_dispatch': 3, 'research_wait': 3, 'research_constructor_span': 1, 'research_before_enter': 1, 'research_after_enter': 1, 'research_manual_span': 1, 'research_manual_work': 1, 'research_exception_span': 1}
[{'args': {'step_num': '0'}, 'inclusive_us': 248.431, 'selected_children_union_us': 243.91500000000002, 'remaining_scope_us': 4.515999999999991}, {'args': {'step_num': '1'}, 'inclusive_us': 223.332, 'selected_children_union_us': 219.96599999999995, 'remaining_scope_us': 3.3660000000000423}, {'args': {'step_num': '2'}, 'inclusive_us': 282.768, 'selected_children_union_us': 280.47499999999997, 'remaining_scope_us': 2.2930000000000064}]


## 编译 pass 与 Python tracing

冷编译同线程范围内的事件不等于全线程编译图，也不是同样数量的独立 pass。
函数体的 host marker 只随 tracing 执行；三次 executable 执行需单独标记。

In [4]:
compiler = results["compiler"]["summary"]
print({k: compiler[k] for k in ("trace_count", "python_inside_jit_events", "warm_execution_scopes", "compile_thread_child_events")})
print({name: compiler["compile_event_names"][name] for name in compiler["observed_known_pass_names"]})

{'trace_count': 1, 'python_inside_jit_events': 1, 'warm_execution_scopes': 3, 'compile_thread_child_events': 171}
{'algsimp': 3, 'constant_folding': 2, 'layout-assignment': 1}


## Mosaic 设备标记：IR 操作与边界

下面只展示生产 lowering 生成的标记操作。独立验证器解析 MLIR 检查块内配对。
不能将这些 Mosaic MLIR 操作称为 LLO，也不能据此推断真机事件可见性。

In [5]:
mosaic = results["mosaic"]["summary"]
print(mosaic["phase"])
print(mosaic["cases"])
for line in (base / "mosaic-events-001/named/mosaic-raw.mlir").read_text().splitlines():
    if '"tpu.trace_start"' in line or '"tpu.trace_stop"' in line:
        print(line)
assert not mosaic["backend_compile"] and not mosaic["tpu_execution"] and not mosaic["llo_captured"]

open-source TPU lowering on a CPU host; no backend compilation
[{'case': 'plain', 'trace_start': [], 'trace_stop_count': 0, 'outer_custom_call': 'tpu_custom_call'}, {'case': 'named', 'trace_start': [{'message': '"research_kernel"', 'level': '10 : i32'}, {'message': '"research_dot"', 'level': '10 : i32'}, {'message': '"research_store"', 'level': '10 : i32'}], 'trace_stop_count': 3, 'outer_custom_call': 'tpu_custom_call'}]
    "tpu.trace_start"() <{level = 10 : i32, message = "research_kernel"}> : () -> () loc(#loc18)
    "tpu.trace_start"() <{level = 10 : i32, message = "research_dot"}> : () -> () loc(#loc18)
    "tpu.trace_stop"() : () -> () loc(#loc22)
    "tpu.trace_start"() <{level = 10 : i32, message = "research_store"}> : () -> () loc(#loc22)
    "tpu.trace_stop"() : () -> () loc(#loc)
    "tpu.trace_stop"() : () -> () loc(#loc)


## 自定义 pass 的负对照与构建身份

[验收说明](pass-event-acceptance.md) 分开记录旧 wheel 负对照、匹配源码 baseline、补丁加载和回滚。下面复查真实 trace 与运行中构建的拒绝快照；当前没有自定义 C++ 事件。

In [6]:
from verify_pass_events import verify_pair, verify_rejection
pair = verify_pair(base / "pass-events-unpatched-003", base / "pass-events-unpatched-filtered-003")
for name in ("default", "filtered"):
    item = pair[name]
    print(name, item["observations"]["generic_known_pass_counts"],
          item["observations"]["custom_event_count"], item["max_absolute_errors"])
assert not pair["patched_runtime_pair_accepted"]
print(verify_rejection(base / "pass-events-running-build-rejected-002"))

default {'algsimp': 3, 'constant_folding': 2, 'layout-assignment': 1} 0 [5.551717559629243e-09, 5.551717559629243e-09, 5.551717559629243e-09]
filtered {'constant_folding': 2, 'layout-assignment': 1} 0 [5.551717559629243e-09, 5.551717559629243e-09, 5.551717559629243e-09]
{'capture_id': 'pass-events-running-build-rejected-002', 'manifest_sha256': '51e19b4d69ce46e06b3a058b9bdeb829f1bbcf0266ed6eb64053af6f3b57d3f4', 'artifact_count': 6, 'evidence_level': 'SOURCE-ONLY', 'status_at_read': 'running', 'runtime_accepted': False}


## 新进程重跑默认 / 过滤对照

下面确实启动两个独立 Python 进程，每次使用新目录并保留原始 capture。仍使用当前旧 wheel 的 `absent` 模式，不能替代将来的补丁正对照，也不以这次采集时间作性能比较。

In [7]:
import os, subprocess, uuid
run_id = uuid.uuid4().hex[:12]
paths = [base / f"pass-events-notebook-{run_id}-{name}" for name in ("default", "filtered")]
child_env = {k: v for k, v in os.environ.items() if k != "XLA_FLAGS"}
for i, path in enumerate(paths):
    command = [sys.executable, "-B", str(root / "research/software-stack/pass_events_probe.py"),
               "--output", str(path), "--expected-events", "absent"]
    if i:
        command += ["--disable-pass", "algsimp"]
    result = subprocess.run(command, cwd=root, env=child_env, text=True, capture_output=True)
    assert result.returncode == 0, result.stdout + result.stderr
    print(result.stdout.strip())
fresh = verify_pair(*paths)
assert not fresh["patched_runtime_pair_accepted"]
print({"fresh_captures": [p.name for p in paths], "custom_counts":
       [fresh[k]["observations"]["custom_event_count"] for k in ("default", "filtered")]})

{"capture": "pass-events-notebook-d0247ac6c2db-default", "custom_events": 0, "binding": "unbound-wheel-negative-control", "qualifiers": ["VERSION-SKEW"]}


{"capture": "pass-events-notebook-d0247ac6c2db-filtered", "custom_events": 0, "binding": "unbound-wheel-negative-control", "qualifiers": ["VERSION-SKEW"]}


{'fresh_captures': ['pass-events-notebook-d0247ac6c2db-default', 'pass-events-notebook-d0247ac6c2db-filtered'], 'custom_counts': [0, 0]}


## 待验收

固定源码构建/加载与可逆 compiler 标记补丁；指定 TPU 上的 LLO/设备 trace、数值和标记开销。
实际业务模型与 TPU 条件仍是独立输入，不用本 Notebook 的 CPU 结果代替。